# NTL: bounded runtime optimization of ozertuu Version 37
Original: https://www.kaggle.com/code/ozertuu/0-689-dual-svc-residual-refinement-apex
Reproduction and runtime optimization for liuqingmu096, 2026-11-16. Published pretrained assets and attribution are preserved; no new training is claimed.
Changes: remove complementary stack, Gaussian training-set readout, GSQ and GLM likelihood stages; use full-data adapter plus folds 0/1 for each main family, and Native folds 0/1/2. Cache shared semantic groups.
Write a real prefit baseline first, then atomic checkpoints after each successful model; independently bound all model subprocess trees. Overall inference target 7.5h and parent watchdog 7.8h.
Public runs perform genuine inference on the sample plus 72 length-stratified timing texts. This is a throughput check, not an unbiased accuracy estimate. Official scores come only from completed competition submissions.


## Original author and community credits (quoted source documentation)
# 🏆 SOTA 0.696 | Clean LoRA 0.684 Base + V1 GLM-5.3 Likelihood Graft on 2×T4

> **Competition:** [Name That LLM (NeurIPS 2024 / Kaggle)](https://www.kaggle.com/competitions/name-that-llm)  
> **Score:** **`0.696` Public Leaderboard SOTA** (`0.684` Atlas V9 Clean LoRA + Qwen-3-Embedding-8B + GSQ-27B Base + Empirically Calibrated GLM-5.3 / GLM-5.3-Flash Contrastive Token Likelihood Reranker)  
> **Hardware:** Dual Kaggle NVIDIA T4 GPUs · 100% Offline · Zero Internet · Strict 8.8h Budget Guard  
>
> **Credits & Acknowledgements:**  
> - **[@feiyu26](https://www.kaggle.com/feiyu26)** for [`ntl-0684-v1-graft`](https://www.kaggle.com/code/feiyu26/ntl-0684-v1-graft), which successfully grafted the V1 likelihood stage onto the fast `0.684` S10 stack with empirically measured T4 per-token cost constants (`row_s_flash=0.0225`, `row_s_glm53=0.0442`) and a `7.5h / 8.8h` runtime budget split to eliminate 9-hour rerun timeouts.  
> - **[@estopadilla](https://www.kaggle.com/estopadilla)** (José Padilla) for engineering the sequential layer-segmented GGUF engine (`ntl-segrun`) and contrastive GLM-5.3 / GLM-5.3-Flash likelihood stage.  
> - **[@ozertuu](https://www.kaggle.com/ozertuu)** & the open Kaggle community (**nuridudak**, **concyclics / CHEN Han**, **lightningv08**, **dhruvtibarewal**, **Joseph Adamski**, **Lucas Boesen**, **Nishant Dahal**, **alexchilton**) for the fast `0.684` S10 Clean LoRA + Qwen-3-Embedding Multidepth + GSQ-27B backbone.

---

## 📌 Architecture Overview: Why This Reaches 0.696 Cleanly Within 9 Hours
1. **Fast S10 Base Stack (`RUN_LIMIT_S = 7.5h`):** Uses the proven `0.684` Clean LoRA + Qwen-3-Embedding-8B (40% share) + GSQ-27B (3% share) + Complementary Stack (15% logit weight) pipeline (`infer.py --save_final_logits /kaggle/temp/final-logits.npz`), finishing well ahead of slower 6.7h variants.
2. **V1 GLM-5.3 & GLM-5.3-Flash Likelihood Graft (`V1_DEADLINE_H = 8.8h`):** Ranks test passages by their S10 probability mass on `glm-5.3 + glm-5.3-flash`, selects only the prefix that fits safely inside the remaining wall-clock budget using measured T4 throughput constants, and re-scores with `ntl-segrun` before blending via temperature-matched log-probabilities.
3. **Commit-Time GPU Smoke Verification:** Validates binary SHA-256, CUDA 13 libraries (`libcublasLt.so.13`), all 6 + 19 GGUF segments, and an actual Dual-T4 GPU forward pass (`--split-mid`) before submission.

---

## 📌 Executive Summary & Architecture Breakthrough
This notebook presents the definitive, production-grade implementation of the **0.684 SOTA** solution for *Name That LLM*. By combining multi-depth hidden representations across open-weights LLMs, robust stylometric residual correction, and balanced Sinkhorn assignment, this pipeline achieves top-tier balanced accuracy across 18 target LLM classes.

### 🌟 Key Architectural Pillars:
1. **Ablation of Destructive Adapters:** Complete removal of the legacy `v1` adapter prevents negative gradient interference and preserves sharp margin separation.
2. **Optimal Dual-LoRA Blending:** Calibrated $72\% / 28\%$ weighted fusion between `ntl-q14-lora-v4` and `lora_qwen3_14b` with unnormalized logit accumulation.
3. **Qwen-3-Embedding Multidepth Feature Head (40% Geometric Weight):** Incorporating 5-fold cross-validated layer representations across multi-depth transformer blocks $[2, 4, 6, 8, 12, 16, 20, 24]$.
4. **GSQ-27B Deep Calibrator (3% Weight):** High-capacity parameter scaling utilizing Qwen-3.8-27B quantized representations for complex boundary resolution.
5. **Stylometric Complementary Stack:** DeBERTa-v3 token-level discriminator + sub-word character/word/shape TF-IDF n-grams + LinearSVC ensemble.
6. **Sinkhorn-Knopp Optimal Transport Margin Balancing:** Dynamic entropic regularization enforcing balanced class priors across the test set.

| Component | Model Backbone | Representation / Head | Ensemble Weight |
| :--- | :--- | :--- | :--- |
| **Qwen-3-14B LoRA V5** | `qwen-3-14b-base` | Multidepth [2..24] + Twin Head + Gauss | $3.0\times$ |
| **Qwen-3-8B LoRA V5** | `qwen-3-8b-base` | Multidepth [2..24] + Twin Head + Gauss | $2.0\times$ |
| **LLaMA-3.1-8B LoRA V5**| `llama-3.1-8b-base` | Multidepth [2..24] + Twin Head | $1.0\times$ |
| **N14B Clean LoRA** | `qwen-3-14b-base` | OQ4 ($0.72$) + N14B ($0.28$) Blend | $2.5\times$ |
| **Qwen-2.5-14B LoRA** | `qwen-2.5-14b` | Multidepth + Twin Head | $1.0\times$ |
| **Qwen-3-Embedding Native**| `qwen-3-embedding-8b` | 5-Fold Multidepth Calibrated Head | **40% Share** |
| **GSQ-27B Deep Engine** | `qwen-3.8-27b` | GSQ 120k Calibrated Feature Head | **3% Share** |
| **Complementary Stack** | `deberta-v3-large` + TFIDF | Stylometric LinearSVC + N-gram Ensemble | $15\%$ Logit Weight |

In [ ]:
import json, os, pathlib, subprocess, sys
FILES = {'budget.py': '"""Bound subprocess trees and preserve completed prediction checkpoints."""\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport threading\nimport time\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temp = path.with_name(path.name + \'.tmp\')\n    temp.write_text(json.dumps(value, indent=2), encoding=\'utf-8\')\n    os.replace(temp, path)\n\n\ndef kill_tree(proc):\n    assert proc.pid > 1 and proc.pid != os.getpid()\n    if proc.poll() is not None:\n        return\n    if os.name == \'posix\':\n        # A nested stage owns its own session. Follow OS parent-child links as\n        # well as the outer process group so a nested watchdog cannot escape.\n        def descendants(pid):\n            result=[]\n            try:\n                tasks=list(Path(f\'/proc/{pid}/task\').iterdir())\n            except (FileNotFoundError,PermissionError):\n                return result\n            children=set()\n            for task in tasks:\n                try:children.update(map(int,(task/\'children\').read_text().split()))\n                except (FileNotFoundError,PermissionError):pass\n            for child in children:\n                result.extend(descendants(child));result.append(child)\n            return result\n        for pid in descendants(proc.pid):\n            try:os.kill(pid,signal.SIGKILL)\n            except ProcessLookupError:pass\n        try:os.killpg(proc.pid, signal.SIGKILL)\n        except ProcessLookupError:pass\n    else:\n        # Local Windows tests have a single child. Production Kaggle is POSIX;\n        # only its process-group branch is used for nested model workers.\n        proc.kill()\n    proc.wait(timeout=20)\n\n\ndef run_stage(name, cmd, seconds, output_dir, env=None):\n    """A wall-clock limit covers reads, inference, and all nested subprocesses."""\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    if seconds <= 0:\n        return dict(name=name, status=\'skipped_budget\', seconds=0.0)\n    flags = {\'start_new_session\': True} if os.name == \'posix\' else {\'creationflags\': subprocess.CREATE_NEW_PROCESS_GROUP}\n    log_path = output_dir / (name + \'.log\')\n    with log_path.open(\'w\', encoding=\'utf-8\') as log:\n        stream=name==\'pipeline\'\n        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE if stream else log,\n            stderr=subprocess.STDOUT, env=env, text=True, bufsize=1, **flags)\n        pump=None\n        if stream:\n            def forward():\n                for line in proc.stdout:\n                    log.write(line);log.flush()\n                    print(line,end=\'\',flush=True)\n            pump=threading.Thread(target=forward,daemon=True);pump.start()\n        try:\n            code = proc.wait(timeout=seconds)\n            status = \'complete\' if code == 0 else \'failed\'\n        except subprocess.TimeoutExpired:\n            kill_tree(proc)\n            code, status = proc.returncode, \'timeout\'\n        if pump is not None:pump.join(timeout=5)\n    result = dict(name=name, status=status, returncode=code, seconds=time.monotonic()-started, limit_seconds=seconds)\n    atomic_json(output_dir / (name + \'.status.json\'), result)\n    tail = log_path.read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()[-8:]\n    print(json.dumps(result), flush=True)\n    print(\'\\n\'.join(tail), flush=True)\n    return result\n', 'ensemble.py': '"""Label-safe fusion and atomic, fully validated submission checkpoints."""\nimport os\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nLABELS = [\'claude-fable-5\',\'claude-opus-4-6\',\'claude-opus-4-8\',\'claude-opus-5\',\'claude-sonnet-5\',\n \'gemini-3.1-pro-preview\',\'gemini-3.6-flash\',\'gemini-3.7-flash\',\'gemini-3.8-flash\',\'glm-5.3\',\'glm-5.3-flash\',\n \'gpt-5.5\',\'gpt-5.6-luna\',\'gpt-5.6-sol\',\'gpt-5.6-terra\',\'gpt-6-astra\',\'kimi-k3\',\'qwen3.8-max\']\n\n\ndef softmax(z):\n    z = np.asarray(z, dtype=np.float64)\n    z = z-z.max(1, keepdims=True)\n    p = np.exp(z)\n    return p/p.sum(1, keepdims=True)\n\n\ndef decide_scores(p, prior, strength=.30, target_mix=.35):\n    q = p.copy()\n    marg = np.clip(p.mean(0), 1e-6, None)\n    marg /= marg.sum()\n    prior = np.asarray(prior, dtype=np.float64)\n    prior = prior/prior.sum()\n    target = np.clip((1-target_mix)*marg+target_mix*prior, 1e-6, None)\n    target /= target.sum()\n    for _ in range(100):\n        q *= len(q)*target[None, :]/np.clip(q.sum(0, keepdims=True), 1e-12, None)\n        q /= np.clip(q.sum(1, keepdims=True), 1e-12, None)\n    return (1-strength)*np.log(p+1e-12)+strength*np.log(q+1e-12)\n\n\ndef combine(terms, prior):\n    assert terms and all(v.ndim == 2 and v.shape[1] == 18 and np.isfinite(v).all() for v in terms.values())\n    active = sum(float(abs(v).max()) > 0 for v in terms.values())\n    return decide_scores(softmax(sum(terms.values())/max(1, active)), prior)\n\n\ndef load_member(path, ids):\n    with np.load(path, allow_pickle=False) as z:\n        assert z[\'ids\'].astype(str).tolist() == list(map(str, ids)), \'member row order mismatch\'\n        assert z[\'labels\'].astype(str).tolist() == LABELS, \'member label order mismatch\'\n        values = {k[5:]:z[k].copy() for k in z.files if k.startswith(\'term_\')}\n    assert values and all(v.shape == (len(ids),18) and np.isfinite(v).all() for v in values.values())\n    return values\n\n\ndef checkpoint(path, ids, scores):\n    ids = list(map(str, ids))\n    assert len(ids) and len(set(ids)) == len(ids)\n    scores = np.asarray(scores)\n    assert scores.shape == (len(ids),18) and np.isfinite(scores).all()\n    frame = pd.DataFrame({\'id\':ids, \'model\':np.asarray(LABELS)[scores.argmax(1)]})\n    path = Path(path)\n    temp = path.with_name(path.name+\'.tmp.parquet\')\n    frame.to_parquet(temp, index=False)\n    assert pd.read_parquet(temp).equals(frame)\n    os.replace(temp,path)\n    return frame\n', 'gauss.py': 'import itertools\nimport numpy as np\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n', 'infer_worker.py': 'import os, sys, time, gc, argparse, json, re\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport numpy as np, pandas as pd, scipy.sparse as sp, torch\nfrom pipe import pipelined, spans_devices\n\nLABELS = [\'claude-fable-5\',\'claude-opus-4-6\',\'claude-opus-4-8\',\'claude-opus-5\',\'claude-sonnet-5\',\n \'gemini-3.1-pro-preview\',\'gemini-3.6-flash\',\'gemini-3.7-flash\',\'gemini-3.8-flash\',\'glm-5.3\',\'glm-5.3-flash\',\n \'gpt-5.5\',\'gpt-5.6-luna\',\'gpt-5.6-sol\',\'gpt-5.6-terra\',\'gpt-6-astra\',\'kimi-k3\',\'qwen3.8-max\']\nL2I = {l: i for i, l in enumerate(LABELS)}\n\np = argparse.ArgumentParser()\np.add_argument(\'--data\', required=True)\np.add_argument(\'--models\', required=True)\np.add_argument(\'--out\', default=\'submission.parquet\')\np.add_argument(\'--save_final_logits\', default=\'\')\np.add_argument(\'--comp_path\', default=\'\')\np.add_argument(\'--comp_w\', type=float, default=0.0)\np.add_argument(\'--tfidf_w\', type=float, default=1.0)\np.add_argument(\'--distort_w\', type=float, default=0.0)\np.add_argument(\'--adapt_thr\', type=float, default=0.15)\np.add_argument(\'--adapt_w\', type=float, default=0.0)\np.add_argument(\'--extra\', default=\'\')\np.add_argument(\'--extra_w\', type=float, default=0.5)\np.add_argument(\'--svc_w\', type=float, default=0.0)\np.add_argument(\'--pool\', default=\'\')\np.add_argument(\'--sk_div\', type=int, default=1)\np.add_argument(\'--sinkhorn\', type=float, default=0.0)\np.add_argument(\'--sinkhorn_target_mix\', type=float, default=0.35)\np.add_argument(\'--atlas_dir\', default=\'\')\np.add_argument(\'--atlas_base\', default=\'\')\np.add_argument(\'--atlas_mode\', default=\'off\', choices=[\'off\', \'head\', \'lite\', \'mix\'])\np.add_argument(\'--atlas_member_w\', type=float, default=0.0)\np.add_argument(\'--atlas_k\', type=int, default=6)\np.add_argument(\'--atlas_w\', type=float, default=0.0)\np.add_argument(\'--atlas_margin\', type=float, default=0.0)\np.add_argument(\'--atlas_bs\', type=int, default=4)\np.add_argument(\'--atlas_chunk\', type=int, default=3)\np.add_argument(\'--atlas_workers\', type=int, default=2)\np.add_argument(\'--atlas_deadline\', type=float, default=0.0)\np.add_argument(\'--save_lp\', default=\'\')\np.add_argument(\'--native_dir\', default=\'\')\np.add_argument(\'--native_base\', default=\'\')\np.add_argument(\'--native_share\', type=float, default=0.0)\np.add_argument(\'--native_cent\', type=int, default=1)\np.add_argument(\'--native_deadline\', type=float, default=0.0)\np.add_argument(\'--optional_deadline\', type=float, default=0.0)\np.add_argument(\'--native_work\', default=\'/kaggle/temp/native\')\np.add_argument(\'--native_min_agree\', type=float, default=0.35)\np.add_argument(\'--prefit_core\', default=\'\')\np.add_argument(\'--gsq_dir\', default=\'\')\np.add_argument(\'--gsq_share\', type=float, default=0.0)\np.add_argument(\'--gsq_deadline\', type=float, default=0.0)\np.add_argument(\'--gsq_timeout\', type=float, default=0.0)\np.add_argument(\'--gsq_work\', default=\'/kaggle/temp/gsq\')\np.add_argument(\'--gsq_min_agree\', type=float, default=0.30)\np.add_argument(\'--group_embed_base\', default=\'\')\np.add_argument(\'--group_embed_assets\', default=\'\')\np.add_argument(\'--group_work\', default=\'/kaggle/temp/qgroups\')\np.add_argument(\'--group_sim\', type=float, default=0.70)\na = p.parse_args()\ntorch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\nT0 = time.time()\ndef log(*x): print(f\'[{time.time()-T0:7.0f}s]\', *x, flush=True)\n\ntrain = pd.read_csv(os.path.join(a.data, \'train.csv\'))\ntest = pd.read_csv(os.path.join(a.data, \'test.csv\'))\ntrain[\'text\'] = train.text.fillna(\'\'); test[\'text\'] = test.text.fillna(\'\')\ny = train.model.map(L2I).values; Gtr = train.prompt_id.values\nlog(\'train\', len(train), \'test\', len(test))\n\ndef capped_agglo(S, cap=18, thr=0.02):\n    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)\n    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)\n    while True:\n        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]\n        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)\n        if not np.isfinite(M[i, j]) or M[i, j] < thr: break\n        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])\n        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf\n        size[i] += size[j]; alive[j] = False; lab[lab == j] = i\n    return lab\n\nfrom sklearn.feature_extraction.text import TfidfVectorizer\ngroup_output = os.environ.get(\'NTL_GROUP_OUTPUT\', \'\')\nif group_output and os.path.isfile(group_output):\n    Gte = np.load(group_output, allow_pickle=False)\nelif os.environ.get(\'NTL_GROUP_EMBED\'):\n    E_q = np.load(os.environ[\'NTL_GROUP_EMBED\'], allow_pickle=False).astype(np.float32)\n    assert E_q.shape[0] == len(test) and np.isfinite(E_q).all()\n    Gte = capped_agglo(E_q @ E_q.T, 18, .70)\nelse:\n    try:\n        v = TfidfVectorizer(sublinear_tf=True, stop_words=\'english\', min_df=2, max_df=.05)\n        Xc = v.fit_transform(test.text.values)\n        Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, .15)\n    except ValueError:\n        Gte = np.arange(len(test))\nassert Gte.shape == (len(test),)\nif group_output:\n    np.save(group_output, Gte, allow_pickle=False)\nlog(\'shared groups\', len(np.unique(Gte)))\nLP = {}\nEXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)\nEXTRA_DF = pd.DataFrame(columns=[\'text\', \'model\'])\ndef _norm_text_key(value):\n    import unicodedata\n    return re.sub(r\'\\s+\', \' \', unicodedata.normalize(\'NFKC\', str(value or \'\'))).strip()\nif a.extra:\n    _E = pd.read_parquet(a.extra) if a.extra.endswith(\'.parquet\') else pd.read_csv(a.extra)\n    if not {\'text\', \'model\'}.issubset(_E.columns):\n        raise ValueError(f\'External file must contain text/model columns: {a.extra}\')\n    _E = _E[_E.model.isin(LABELS)].copy()\n    _E[\'text\'] = _E.text.fillna(\'\').astype(str)\n    _official_keys = {_norm_text_key(t) for t in train.text.values}\n    _seen = set(_official_keys)\n    _keep = []\n    for _t in _E.text.values:\n        _k = _norm_text_key(_t)\n        _ok = bool(_k) and _k not in _seen\n        _keep.append(_ok)\n        if _ok: _seen.add(_k)\n    EXTRA_DF = _E.loc[_keep, [\'text\', \'model\']].reset_index(drop=True)\n    EXT_T = EXTRA_DF.text.astype(str).values\n    EXT_Y = EXTRA_DF.model.map(L2I).values.astype(int)\n    log(\'matched-clean extra rows after exact de-dup\', len(EXT_T))\nFIT_T = np.concatenate([train.text.values, EXT_T])\nFIT_Y = np.concatenate([y, EXT_Y])\nFIT_W = np.concatenate([np.ones(len(train), dtype=np.float32), np.full(len(EXT_T), a.extra_w, dtype=np.float32)])\nif a.tfidf_w > 0:\n    from sklearn.linear_model import LogisticRegression\n    mats_tr, mats_te = [], []\n    for kw in [dict(analyzer=\'char_wb\', ngram_range=(2, 5), min_df=3, max_features=60000),\n               dict(token_pattern=r"\\w+|[^\\w\\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:\n        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T)\n        mats_tr.append(vec.transform(FIT_T)); mats_te.append(vec.transform(test.text.values))\n    LP[\'tfidf\'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w\n    gc.collect(); log(\'tfidf done\')\nelse:\n    LP[\'tfidf\'] = np.zeros((len(test), 18), dtype=np.float32)\n    log(\'tfidf skipped (zero ensemble weight)\')\nif a.svc_w > 0:\n    import re as _re\n    from scipy.sparse import hstack as _hs\n    from sklearn.svm import LinearSVC\n    _FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())\n    _shape = lambda t: _re.sub(r"[^\\W\\d_]+", lambda m_: "X" if m_[0].isupper() else "x", t)\n    _func = lambda t: " ".join(w.lower() if w.lower() in _FW or not w.isalnum() else "CONTENT" for w in _re.findall(r"\\w+|[^\\w\\s]", t))\n    _parts = []\n    _official = train[[\'text\', \'model\']].copy(); _official[\'text\'] = _official.text.fillna(\'\').astype(str)\n    _official[\'_srcw\'] = 1.0\n    _parts.append(_official)\n    _seen = {_norm_text_key(t) for t in _official.text.values}\n    if len(EXTRA_DF):\n        _e = EXTRA_DF[[\'text\', \'model\']].copy(); _e[\'_srcw\'] = float(a.extra_w)\n        _keep = []\n        for _t in _e.text.values:\n            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen\n            _keep.append(_ok)\n            if _ok: _seen.add(_k)\n        _e = _e.loc[_keep].copy(); _parts.append(_e)\n    if a.pool and os.path.exists(a.pool):\n        _P = pd.read_parquet(a.pool)\n        if not {\'text\', \'model\'}.issubset(_P.columns):\n            raise ValueError(f\'Pool must contain text/model columns: {a.pool}\')\n        _P = _P[_P.model.isin(LABELS)][[\'text\', \'model\']].copy()\n        _P[\'text\'] = _P.text.fillna(\'\').astype(str); _P[\'_srcw\'] = 0.35\n        _keep = []\n        for _t in _P.text.values:\n            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen\n            _keep.append(_ok)\n            if _ok: _seen.add(_k)\n        _P = _P.loc[_keep].copy(); _parts.append(_P)\n    _fitdf = pd.concat(_parts, ignore_index=True)\n    _fit = _fitdf.text.astype(str).tolist(); _fy = _fitdf.model.map(L2I).to_numpy(dtype=int); _sw = _fitdf[\'_srcw\'].to_numpy(dtype=np.float32)\n    log(f\'svc4 fit rows={len(_fit)} source_weighted; external clean={len(EXTRA_DF)}\')\n    _A, _B = [], []\n    for view, lo, hi, mf, wv in [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]:\n        prep = _shape if view == "shape" else _func if view == "function" else (lambda s_: s_)\n        an = "char" if view == "shape" else "word" if view == "function" else view\n        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)\n        if an == "word": o["token_pattern"] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n        _v = TfidfVectorizer(**o); _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv); _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)\n    _svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight=\'balanced\')\n    _svc.fit(_hs(_A, format=\'csr\'), _fy, sample_weight=_sw)\n    _d = _svc.decision_function(_hs(_B, format=\'csr\'))\n    LP[\'svc4\'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w\n    del _A, _B, _parts, _fitdf, _svc; gc.collect(); log(\'svc4 done\')\nif a.distort_w > 0:\n    import re, collections\n    wre = re.compile(r"[A-Za-z’\']+")\n    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))\n    keep = {w for w, _ in cnt.most_common(100)}\n    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else \'*\', t) for t in arr]\n    Dtr, Dte = dist(FIT_T), dist(test.text.values)\n    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer=\'char\', ngram_range=(2, 6), min_df=2),\n          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\\s]+", ngram_range=(1, 3), min_df=2)]\n    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()\n    LP[\'distort\'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w\n    log(\'distort done\')\nif a.prefit_core:\n    from ntl660_readout import core_votes\n    _pv = core_votes(a.prefit_core, test.text.tolist())\n    assert set(_pv) == {\'svc4\', \'tfidf\', \'distort\'} and all(v.shape == (len(test), 18) and np.isfinite(v).all() for v in _pv.values())\n    LP.update(_pv)\n    log(\'prefit core votes loaded\', {k: round(float(abs(v).max()), 2) for k, v in _pv.items()})\nfrom transformers import AutoTokenizer, AutoModel\n@torch.no_grad()\ndef l24_predict(spec, texts, groups=None, train_texts=None):\n    from transformers import AutoConfig\n    from safetensors.torch import load_file\n    from concurrent.futures import ThreadPoolExecutor\n    adapters = spec.get(\'adapters\') or [spec[\'adapter\']]\n    aw = spec.get(\'adapter_weights\')\n    if aw is None or len(aw) != len(adapters): aw = [1.0 / len(adapters)] * len(adapters)\n    tok = AutoTokenizer.from_pretrained(spec[\'path\'])\n    if tok.pad_token is None: tok.pad_token = tok.eos_token\n    cfg = AutoConfig.from_pretrained(spec[\'path\']); tc = cfg.text_config if hasattr(cfg, \'text_config\') else cfg\n    tc.num_hidden_layers = spec[\'keep\']; tc.use_cache = False\n    if getattr(tc, \'layer_types\', None) is not None: tc.layer_types = tc.layer_types[:spec[\'keep\']]\n    _ms = []\n    if spec.get(\'dp\') and torch.cuda.device_count() >= 2:\n        try:\n            for _d in range(2):\n                _ms.append(AutoModel.from_pretrained(spec[\'path\'], config=cfg, dtype=torch.float16, device_map={\'\': _d}).eval())\n                for _n in (\'vision_tower\', \'multi_modal_projector\'):\n                    if getattr(_ms[-1], _n, None) is not None: setattr(_ms[-1], _n, None)\n                gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(_d)\n                _ms[-1](input_ids=torch.full((spec.get(\'bs\', 16), spec.get(\'max_len\', 384)), tok.pad_token_id, device=f\'cuda:{_d}\'), output_hidden_states=True, use_cache=False)\n                assert torch.cuda.max_memory_allocated(_d) < torch.cuda.get_device_properties(_d).total_memory - (1 << 30), \'under 1 GiB to spare\'\n        except Exception as _e:\n            log(f\'  l24 {spec["name"]}: one split copy instead of one per GPU ({_e!r:.80})\'); _ms = []\n        gc.collect(); torch.cuda.empty_cache()\n    if not _ms: _ms = [AutoModel.from_pretrained(spec[\'path\'], config=cfg, dtype=torch.float16, device_map=\'auto\').eval()]\n    m = _ms[0]\n    if spec.get(\'no_final_norm\'):\n        for _m in _ms: _m.norm = torch.nn.Identity()\n    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec[\'layers\']\n    _mods = [dict(_m.named_modules()) for _m in _ms]\n    _eos = spec.get(\'eos_id\')\n    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get(\'max_len\', 384) - (1 if _eos is not None else 0))[\'input_ids\']]\n    enc = _tk(texts)\n    order = np.argsort([len(x) for x in enc]); bs = spec.get(\'bs\', 16)\n    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))\n    P_acc, C_acc = 0.0, 0.0; FEATS = None; OVF = [0, 0]\n    if train_texts is not None:\n        enc_tr = _tk(train_texts)\n    def featurize(enc_, norms, hid):\n        order_ = np.argsort([len(x) for x in enc_])\n        Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)\n        batches = [order_[i:i + bs] for i in range(0, len(order_), bs)]\n\n        def feats(mm, b, check=True):\n            dev = next(mm.parameters()).device\n            width = max(len(enc_[j]) for j in b)\n            ids = torch.full((len(b), width), tok.pad_token_id)\n            att = torch.zeros((len(b), width), dtype=torch.long)\n            for r, j in enumerate(b):\n                ids[r, :len(enc_[j])] = torch.tensor(enc_[j])\n                att[r, :len(enc_[j])] = 1\n            with torch.no_grad():\n                hs = mm(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states\n                if check and not all(torch.isfinite(hs[l]).all() for l in L):\n                    return None\n                mask = att.unsqueeze(-1).float()\n                mask[:, 0] = 0\n                mg = {d_: mask.to(d_) for d_ in {hs[l].device for l in L}}\n                return torch.cat([nm(((hs[l].float() * mg[hs[l].device]).sum(1) / mg[hs[l].device].sum(1)).cpu()) for l, nm in zip(L, norms)], -1).numpy()\n\n        def put(b, r):\n            Z_[b] = r\n\n        def checked(k, b):\n            mm = _ms[k]\n            r = feats(mm, b)\n            if r is None:\n                mm.to(torch.bfloat16)\n                r = feats(mm, b, check=False)\n                mm.to(torch.float16)\n                OVF[k] += 1\n            put(b, r)\n\n        def run_copy(k, n):\n            for b in batches[k::n]:\n                checked(k, b)\n\n        if len(_ms) == 1:\n            if spans_devices(_ms[0]):\n                pipelined(batches, lambda b: feats(_ms[0], b), put, lambda b: checked(0, b), workers=2, tag=\'l24 \' + spec[\'name\'], log=log)\n            else:\n                for b in batches:\n                    checked(0, b)\n        else:\n            try:\n                with ThreadPoolExecutor(len(_ms)) as _ex:\n                    for _f in [_ex.submit(run_copy, k, len(_ms)) for k in range(len(_ms))]:\n                        _f.result()\n            except Exception as _e:\n                log(f\'  l24 {spec["name"]}: two-copy pass failed ({_e!r:.80}), one copy from here on\')\n                del _ms[1:], _mods[1:]\n                gc.collect()\n                torch.cuda.empty_cache()\n                for b in batches:\n                    checked(0, b)\n        return Z_\n    for ai, ad in enumerate(adapters):\n        adp = os.path.join(ad, \'adapter\') if os.path.exists(os.path.join(ad, \'adapter\', \'adapter_config.json\')) else ad\n        acfg = json.load(open(os.path.join(adp, \'adapter_config.json\'))); scale = acfg[\'lora_alpha\'] / acfg[\'r\']\n        sd = load_file(os.path.join(adp, \'adapter_model.safetensors\')); deltas = []\n        for k, v in sd.items():\n            if \'.lora_A.\' not in k: continue\n            name = k.replace(\'base_model.model.\', \'\', 1).split(\'.lora_A.\')[0]\n            if name not in mods and (\'model.\' + name) in mods: name = \'model.\' + name\n            B = sd[k.replace(\'.lora_A.\', \'.lora_B.\')]\n            for _md in _mods:\n                W = _md[name].weight\n                W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)\n                deltas.append((W, B, v))\n        if os.path.exists(os.path.join(ad, \'heads.pt\')): H = torch.load(os.path.join(ad, \'heads.pt\'), map_location=\'cpu\')\n        else: H = dict(load_file(os.path.join(ad, \'heads.safetensors\')))\n        if os.path.exists(os.path.join(ad, \'meta.json\')):\n            _lab = json.load(open(os.path.join(ad, \'meta.json\')))[\'labels\']; _perm = [_lab.index(l) for l in LABELS]\n            for _p in (\'plain\', \'centered\'): H[f\'{_p}.weight\'] = H[f\'{_p}.weight\'][_perm]; H[f\'{_p}.bias\'] = H[f\'{_p}.bias\'][_perm]\n        hid = H[\'norms.0.weight\'].shape[0]\n        norms = [torch.nn.LayerNorm(hid) for _ in L]\n        for i, nm in enumerate(norms): nm.load_state_dict({\'weight\': H[f\'norms.{i}.weight\'], \'bias\': H[f\'norms.{i}.bias\']})\n        Z = featurize(enc, norms, hid)\n        if train_texts is not None and ai == len(adapters) - 1:\n            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))\n        for W, B, A_ in deltas:\n            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)\n        torch.cuda.empty_cache()\n        del deltas, sd; gc.collect()\n        hd = lambda p, X: lsm(X @ H[f\'{p}.weight\'].float().numpy().T + H[f\'{p}.bias\'].float().numpy())\n        P_acc = P_acc + hd(\'plain\', Z) * aw[ai]\n        if groups is not None:\n            Zc = Z.copy()\n            for g in np.unique(groups):\n                ii = groups == g; Zc[ii] -= Zc[ii].mean(0)\n            C_acc = C_acc + hd(\'centered\', Zc) * aw[ai]\n        log(f\'  l24 {spec["name"]}: adapter {os.path.basename(ad)} done (bf16 redo batches {sum(OVF)})\')\n    del m, mods, _ms, _mods; gc.collect(); torch.cuda.empty_cache()\n    if train_texts is not None: return P_acc, (C_acc if groups is not None else None), FEATS\n    return P_acc, (C_acc if groups is not None else None)\n\ndef gpu_mem():\n    return \' \'.join(f\'gpu{i}:{torch.cuda.memory_allocated(i)/1e9:.1f}GB\' for i in range(torch.cuda.device_count()))\n\nspecs = json.loads(a.models)\nfailed = []\n\n\ndef run_l24(spec):\n    log(\'l24\', spec[\'name\'])\n    out = {}\n    if spec.get(\'gauss\', 0) > 0:\n        plain, cent, (Zte_g, Ztr_g) = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)\n        import gauss as _ga\n        prm = _ga.fit(Ztr_g, y, Gtr, dict(red=\'wpca\', d=128, se_shrink=\'lw\', sp_shrink=\'lw\', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))\n        prm[\'T1\'] = prm[\'Tg\'] = 3.0\n        _sz = np.bincount(Gte)[Gte]\n        _rows = (_sz >= 2) & (_sz <= 18)\n        _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm)\n        assert np.allclose(np.exp(_q).sum(1), 1)\n        _G = np.zeros((len(test), 18))\n        if _rows.any():\n            _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)\n        out[spec[\'name\'] + \'_gauss\'] = _G * spec[\'gauss\']\n        log(f"  gaussian head {spec[\'name\']}: {int(_rows.sum())} grouped test rows, weight {spec[\'gauss\']}")\n    else:\n        plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)\n    out[spec[\'name\']] = plain * spec.get(\'w\', 1.0)\n    if cent is not None:\n        rows = np.bincount(Gte)[Gte] >= 2\n        out[spec[\'name\'] + \'_twin\'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get(\'w\', 1.0) * spec.get(\'twin\', 1.0)\n    LP.update(out)\n\n\ndef run_enc(spec):\n    log(\'enc\', spec[\'name\'])\n    from safetensors.torch import load_file as _lf\n    _ck = spec[\'ckpt\']\n    _meta = json.load(open(os.path.join(_ck, \'meta.json\')))\n    _perm = [_meta[\'labels\'].index(l) for l in LABELS]\n    _hd = _lf(os.path.join(_ck, \'head.safetensors\'))\n    _W = _hd[\'weight\'].float().numpy()[_perm]\n    _b = _hd[\'bias\'].float().numpy()[_perm]\n    _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, \'encoder\'))\n    _enc = AutoModel.from_pretrained(os.path.join(_ck, \'encoder\'), dtype=torch.float32).to(\'cuda:0\').eval()\n    _ids = _etok(list(test.text.values), truncation=True, max_length=_meta[\'max_len\'])[\'input_ids\']\n    _ord = np.argsort([len(x) for x in _ids], kind=\'stable\')\n    Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)\n    with torch.inference_mode():\n        for _s in range(0, len(_ord), 32):\n            _idx = _ord[_s:_s + 32]\n            _bt = _etok.pad({\'input_ids\': [_ids[i] for i in _idx]}, return_tensors=\'pt\').to(\'cuda:0\')\n            _m = _bt[\'attention_mask\'].unsqueeze(-1).float()\n            Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()\n    del _enc\n    _zz = Z_ @ _W.T + _b\n    LP[spec[\'name\']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get(\'w\', 1.0)\n\n\nRUNNERS = {\'l24\': run_l24, \'enc\': run_enc}\nfor spec in specs:\n    if spec.get(\'optional\') and a.optional_deadline and time.time() > a.optional_deadline:\n        log(\'MEMBER SKIPPED (time):\', spec[\'name\'])\n        failed.append(spec[\'name\'])\n        continue\n    try:\n        RUNNERS[spec[\'type\']](spec)\n    except Exception as _e:\n        failed.append(spec[\'name\'])\n        log(f"MEMBER FAILED {spec[\'name\']}: {_e!r}")\n    gc.collect()\n    torch.cuda.empty_cache()\n    log(spec[\'name\'], \'done\', gpu_mem())\nlog(\'members failed:\', failed if failed else \'none\')\n\ntot = sum(LP.values())\nassert not failed, failed\nmember_output = os.environ[\'NTL_MEMBER_OUTPUT\']\nmember_terms = {\'term_\'+k:np.asarray(v) for k,v in LP.items() if k not in (\'tfidf\',\'distort\',\'svc4\')}\nassert member_terms and all(v.shape == (len(test),18) and np.isfinite(v).all() for v in member_terms.values())\nnp.savez(member_output+\'.tmp.npz\', ids=test.id.astype(str).values.astype(\'U\'), labels=np.array(LABELS), **member_terms)\nos.replace(member_output+\'.tmp.npz\',member_output)\nif a.save_lp:\n    np.save(a.save_lp, tot)\nCOMP_P = None\nif a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):\n    try:\n        _cm = json.load(open(a.comp_path.replace(\'comp_logit.npy\', \'comp_meta.json\')))\n        assert _cm[\'ids\'] == test[\'id\'].astype(str).tolist(), \'comp stack row order differs\'\n        _cl = np.load(a.comp_path)[:, [_cm[\'labels\'].index(l) for l in LABELS]]\n        COMP_P = np.exp(_cl - _cl.max(1, keepdims=True))\n        COMP_P /= COMP_P.sum(1, keepdims=True)\n        assert np.isfinite(COMP_P).all()\n        log(f"comp stack loaded ({_cm[\'seconds\']:.0f}s)")\n    except Exception as _e:\n        log(\'COMP SKIPPED:\', repr(_e))\n        COMP_P = None\nelif a.comp_w > 0:\n    log(\'COMP SKIPPED: no stack output\')\n\n\ndef blend(terms):\n    total = sum(terms.values())\n    active = sum(abs(v).max() > 0 for v in terms.values())\n    lpn = total / max(1, active) if a.sk_div else total\n    P = np.exp(lpn - lpn.max(1, keepdims=True))\n    P /= P.sum(1, keepdims=True)\n    if COMP_P is not None:\n        P = (1 - a.comp_w) * P + a.comp_w * COMP_P\n        P /= P.sum(1, keepdims=True)\n    return P\n\n\ndef decide_scores(P):\n    Q = P.copy()\n    rho = float(np.clip(a.sinkhorn_target_mix, 0.0, 1.0))\n    marg = np.clip(P.mean(0), 1e-6, None)\n    marg /= marg.sum()\n    prior = np.bincount(y, minlength=18).astype(np.float64) + 1.0\n    prior /= prior.sum()\n    target = np.clip((1.0 - rho) * marg + rho * prior, 1e-6, None)\n    target /= target.sum()\n    for _ in range(100):\n        Q *= (len(Q) * target[None, :]) / np.clip(Q.sum(0, keepdims=True), 1e-12, None)\n        Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)\n    return (1 - a.sinkhorn) * np.log(P + 1e-12) + a.sinkhorn * np.log(Q + 1e-12)\n\n\ndef decide(P):\n    return decide_scores(P).argmax(1)\n\n\ndef save_final_logits(z):\n    if a.save_final_logits:\n        np.savez(a.save_final_logits, ids=test.id.astype(str).values.astype(\'U\'),\n                 labels=np.array(LABELS), logits=np.asarray(z, dtype=np.float64))\n\n\ndef write_pred(pred, tag):\n    sub = pd.DataFrame({\'id\': test.id.values, \'model\': [LABELS[i] for i in pred]})\n    assert sub.model.isin(LABELS).all() and sub.id.is_unique and len(sub) == len(test)\n    sub.to_parquet(a.out, index=False)\n    log(f\'wrote {tag}\', sub.model.value_counts().to_dict())\n\n\ndef write(P, tag):\n    save_final_logits(decide_scores(P))\n    write_pred(decide(P), tag)\n\n\nP = blend(LP)\nwrite(P, \'blend without ATLAS\')\n\n\ndef atlas_lite(k):\n    import atlas_infer\n    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)\n    try:\n        enc = scorer.encode(test.text.values)\n        to_b = np.array([scorer.labels.index(l) for l in LABELS])\n        cand = np.argsort(-P, 1)[:, :k]\n        mean_lp, done = scorer.run_candidates(enc, to_b[cand], a.atlas_deadline or None, workers=a.atlas_workers)\n        lp = scorer.candidate_logprobs(mean_lp[done], to_b[cand][done])\n    finally:\n        scorer.close()\n    rows = np.repeat(lp.min(1, keepdims=True), 18, 1)\n    np.put_along_axis(rows, cand[done], lp, 1)\n    term = np.zeros((len(test), 18))\n    term[done] = rows\n    return term, done\n\n\ndef atlas_scores(rows, calibrate):\n    import atlas_infer\n    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)\n    try:\n        enc = scorer.encode(test.text.values)\n        out, done = scorer.run(enc, rows, deadline=a.atlas_deadline or None, batch=a.atlas_bs, chunk=a.atlas_chunk, workers=a.atlas_workers)\n        order = [scorer.labels.index(l) for l in LABELS]\n        if calibrate:\n            res = np.zeros((len(test), 18))\n            res[done] = scorer.calibrated(out[done])\n        else:\n            raw = out[..., 1:].mean(-1)\n            res = np.divide(raw, raw.sum(1, keepdims=True), out=np.zeros_like(raw), where=raw.sum(1, keepdims=True) > 0)\n    finally:\n        scorer.close()\n    return res[:, order], done\n\n\nif a.atlas_mode != \'off\' and a.atlas_dir:\n    if a.atlas_deadline and time.time() > a.atlas_deadline - 600:\n        log(\'ATLAS SKIPPED: not enough time left\')\n    else:\n        try:\n            if a.atlas_mode == \'lite\':\n                term, done = atlas_lite(a.atlas_k)\n                P = blend({**LP, \'atlas\': a.atlas_member_w * term})\n                log(f\'atlas lite top-{a.atlas_k} on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}\', gpu_mem())\n            elif a.atlas_mode == \'head\':\n                lp, done = atlas_scores(np.arange(len(test)), True)\n                P = blend({**LP, \'atlas\': a.atlas_member_w * lp})\n                log(f\'atlas head on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}\', gpu_mem())\n            else:\n                top2 = np.sort(P, 1)\n                margin = top2[:, -1] - top2[:, -2]\n                rows = np.where(margin < a.atlas_margin)[0] if a.atlas_margin > 0 else np.arange(len(test))\n                A_, done = atlas_scores(rows, False)\n                use = rows[done[rows]]\n                mixed = P.copy()\n                mixed[use] += a.atlas_w * (A_[use] - P[use])\n                P = mixed / mixed.sum(1, keepdims=True)\n                log(f\'atlas mixed on {len(use)}/{len(rows)} rows at weight {a.atlas_w}\', gpu_mem())\n        except Exception as _e:\n            log(\'ATLAS SKIPPED:\', repr(_e))\n        write(P, \'final blend\')\n\n\nif a.gsq_share > 0 and a.gsq_dir:\n    if a.gsq_deadline and time.time() > a.gsq_deadline:\n        log(\'GSQ SKIPPED: not enough time left\')\n    else:\n        try:\n            import gsq_stage as _gs\n            import shutil as _sh\n            _sh.rmtree(\'/tmp/llama-3.1\', ignore_errors=True)\n            gc.collect()\n            torch.cuda.empty_cache()\n            Q27 = _gs.run(a.gsq_dir, test[\'id\'].astype(str).tolist(), test.text.tolist(), Gte, a.gsq_work, LABELS, log, a.gsq_timeout or None)\n            _agree = float((Q27.argmax(1) == P.argmax(1)).mean())\n            log(f\'GSQ 27B agreement with the blend {_agree:.3f}\')\n            if _agree < a.gsq_min_agree:\n                raise RuntimeError(f\'GSQ agreement {_agree:.3f} below {a.gsq_min_agree}: refusing to use it\')\n            P = (1 - a.gsq_share) * P + a.gsq_share * Q27\n            P /= P.sum(1, keepdims=True)\n            write(P, \'blend with GSQ 27B\')\n        except Exception as _e:\n            log(\'GSQ SKIPPED:\', repr(_e))\n\n\ndef rowsoftmax(z):\n    z = z - z.max(1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(1, keepdims=True)\n\n\ndef native_stage(P):\n    import subprocess\n    import native_q8\n    os.makedirs(a.native_work, exist_ok=True)\n    out = os.path.join(a.native_work, \'native.npz\')\n    if os.path.exists(out):\n        os.remove(out)\n    gc.collect()\n    torch.cuda.empty_cache()\n    cmd = [sys.executable, \'-u\', os.path.join(os.path.dirname(os.path.abspath(__file__)), \'native_q8.py\'), \'--base\', a.native_base, \'--assets\', a.native_dir,\n           \'--query\', os.path.join(a.data, \'test.csv\'), \'--out\', out, \'--work\', a.native_work, \'--deadline\', str(a.native_deadline)]\n    env = dict(os.environ, HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\', TOKENIZERS_PARALLELISM=\'false\')\n    code = subprocess.run(cmd, env=env).returncode\n    if not os.path.exists(out):\n        raise RuntimeError(f\'native worker produced nothing (exit code {code})\')\n    z = np.load(out, allow_pickle=False)\n    folds = z[\'folds\'].tolist()\n    assert z[\'ids\'].astype(str).tolist() == test[\'id\'].astype(str).tolist(), \'native row order differs\'\n    labels = z[\'labels\'].astype(str).tolist()\n    order = [labels.index(l) for l in LABELS]\n    states = [json.load(open(os.path.join(a.native_dir, \'calibration\', f\'fold{k}.json\')))[\'models\'][\'qwen8\'] for k in folds]\n    lp = native_q8.calibrate(z[\'plain\'][:, :, order], z[\'centered\'][:, :, order], z[\'centered_bias\'][:, order], Gte, states, bool(a.native_cent))\n    share = (a.native_share * z[\'finite\'])[:, None]\n    logP = np.log(np.maximum(P, 1e-15))\n    agree = [float((lp[i].argmax(1) == P.argmax(1)).mean()) for i in range(len(folds))]\n    log(f\'native folds {folds} (worker exit {code}); agreement with the blend per fold\', [round(v, 3) for v in agree],\n        \'rows without native\', int((~z[\'finite\']).sum()))\n    if min(agree) < a.native_min_agree:\n        raise RuntimeError(f\'native agreement {min(agree):.3f} below {a.native_min_agree}: refusing to use it\')\n    probs = np.zeros_like(P)\n    for i in range(len(folds)):\n        probs += rowsoftmax(decide_scores(rowsoftmax((1 - share) * logP + share * lp[i]))) / len(folds)\n    assert np.isfinite(probs).all() and np.allclose(probs.sum(1), 1)\n    return probs\n\n\nif a.native_share > 0 and a.native_dir and a.native_base:\n    if a.native_deadline and time.time() > a.native_deadline - 1500:\n        log(\'NATIVE SKIPPED: not enough time left\')\n    else:\n        try:\n            probs = native_stage(P)\n            changed = int((probs.argmax(1) != decide(P)).sum())\n            write_pred(probs.argmax(1), \'final blend with native q8\')\n            save_final_logits(np.log(np.maximum(probs, 1e-15)))\n            log(f\'native q8 at share {a.native_share}: {changed}/{len(test)} decisions differ from the blend-only decision\', gpu_mem())\n        except Exception as _e:\n            log(\'NATIVE SKIPPED:\', repr(_e))\n', 'native_q8.py': "import argparse\nimport gc\nimport json\nimport os\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nINSTRUCTION = 'Instruct: Identify the language model that generated the text based on its writing style.\\nQuery: '\nDEPTHS = (3, 6, 9, 12, 18, 24, 30, 36)\nSUFFIXES = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')\nRANK = 16\nSCALE = 2.0\nTOKEN_LIMIT = 512\nTERMINAL = 151643\n\n\ndef log_softmax(z):\n    z = np.asarray(z, dtype=np.float64)\n    z = z - z.max(axis=-1, keepdims=True)\n    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))\n\n\ndef calibrate(plain, centered, bias, groups, selections, use_centered=True):\n    plain, centered, bias = (np.asarray(x, dtype=np.float64) for x in (plain, centered, bias))\n    groups = np.asarray(groups)\n    folds = plain.shape[0]\n    assert plain.shape == centered.shape and plain.ndim == 3 and plain.shape[-1] == 18\n    assert bias.shape == (folds, 18) and groups.shape == (plain.shape[1],) and len(selections) == folds\n    assert all(np.isfinite(x).all() for x in (plain, centered, bias))\n    _, inverse = np.unique(groups, return_inverse=True)\n    count = np.bincount(inverse)\n    eligible = (count[inverse] >= 2) & (count[inverse] <= 18)\n    out = []\n    for k, state in enumerate(selections):\n        assert not state['outer_labels_used'] and state['centered_head_trained']\n        assert state['plain_scale'] > 0 and state['centered_scale'] > 0\n        means = np.zeros((len(count), 18))\n        np.add.at(means, inverse, centered[k])\n        means /= count[:, None]\n        relative = centered[k] - means[inverse] + bias[k]\n        a = log_softmax(plain[k] * state['plain_scale'])\n        b = log_softmax(relative * state['centered_scale'])\n        weight = float(state['selected_centered_weight'])\n        assert 0 <= weight <= 0.5\n        fraction = np.where(eligible & use_centered, weight, 0.0)[:, None]\n        out.append(log_softmax((1 - fraction) * a + fraction * b))\n    return np.stack(out)\n\n\ndef windows(tokenizer, texts):\n    prefix = tokenizer(INSTRUCTION, add_special_tokens=False)['input_ids']\n    assert tokenizer('', add_special_tokens=True)['input_ids'] == [TERMINAL], 'unexpected end-of-text handling'\n    capacity = TOKEN_LIMIT - len(prefix) - 1\n    first_capacity = TOKEN_LIMIT - 1\n    full = tokenizer([INSTRUCTION + str(t) for t in texts], truncation=False, add_special_tokens=False)['input_ids']\n    seqs, owners, weights = [], [], []\n    for row, tokens in enumerate(full):\n        assert len(tokens) > 0\n        seqs.append(tokens[:first_capacity] + [TERMINAL])\n        owners.append(row)\n        weights.append(max(1, min(len(tokens), first_capacity) - len(prefix)))\n        pos = first_capacity\n        while pos < len(tokens):\n            chunk = tokens[pos:pos + capacity]\n            seqs.append(prefix + chunk + [TERMINAL])\n            owners.append(row)\n            weights.append(len(chunk))\n            pos += len(chunk)\n    return seqs, np.asarray(owners), np.asarray(weights, dtype=np.float64)\n\n\nclass NativeQ8:\n    def __init__(self, base, assets, work, log=print, gpus=None):\n        import torch\n        from torch import nn\n        from transformers import AutoModel, AutoTokenizer\n        self.torch, self.nn, self.log = torch, nn, log\n        torch.set_num_threads(2)\n        for key in ('allow_tf32', 'allow_fp16_reduced_precision_reduction', 'allow_bf16_reduced_precision_reduction'):\n            setattr(torch.backends.cuda.matmul, key, False)\n        torch.backends.cudnn.allow_tf32 = False\n        self.assets = Path(assets)\n        self.labels = json.loads((self.assets / 'manifest.json').read_text())['classes']\n        origin = Path(base)\n        index = json.loads((self.assets / 'base' / 'model.safetensors.index.json').read_text())\n        shards = sorted(set(index['weight_map'].values()))\n        small = ['config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json']\n        self.staged = Path(work) / 'native-q8-base'\n        self.staged.mkdir(parents=True, exist_ok=True)\n        for name in small:\n            src = self.assets / 'base' / name\n            assert src.is_file(), str(src)\n            dst = self.staged / name\n            if not dst.exists():\n                shutil.copyfile(src, dst)\n        for name in shards:\n            src, dst = origin / name, self.staged / name\n            assert src.is_file(), str(src)\n            if not dst.exists():\n                try:\n                    os.symlink(src, dst)\n                except OSError:\n                    shutil.copyfile(src, dst)\n        for k in range(5):\n            assert (self.assets / f'fold{k}' / 'trainables.safetensors').is_file(), k\n        cfg = json.loads((self.staged / 'config.json').read_text())\n        self.layers, self.hidden = cfg['num_hidden_layers'], cfg['hidden_size']\n        assert self.layers == 36\n        gpus = gpus if gpus is not None else torch.cuda.device_count()\n        assert gpus >= 1\n        self.head_device = 'cuda:0'\n        place = {'embed_tokens': 0, 'norm': gpus - 1, 'rotary_emb': gpus - 1}\n        cut = 16 if gpus >= 2 else self.layers\n        place.update({f'layers.{i}': 0 if i < cut else 1 for i in range(self.layers)})\n        self.tokenizer = AutoTokenizer.from_pretrained(self.staged, local_files_only=True)\n        self.tokenizer.padding_side = 'right'\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        loaded = AutoModel.from_pretrained(self.staged, local_files_only=True, dtype=torch.float16, device_map=place,\n                                           attn_implementation='sdpa', output_loading_info=True)\n        if isinstance(loaded, tuple):\n            self.model, info = loaded\n            assert not any(info.get(k) for k in ('missing_keys', 'mismatched_keys', 'error_msgs', 'unexpected_keys')), info\n        else:\n            self.model = loaded\n        self.model.config.use_cache = False\n        self.loras = {}\n        for name, mod in list(self.model.named_modules()):\n            if isinstance(mod, nn.Linear) and name.rsplit('.', 1)[-1] in SUFFIXES:\n                wrapped = self._lora(mod)\n                parent, attr = name.rsplit('.', 1)\n                setattr(self.model.get_submodule(parent), attr, wrapped)\n                self.loras[name] = wrapped\n        assert len(self.loras) == self.layers * len(SUFFIXES), len(self.loras)\n        dev = self.head_device\n        self.norms = nn.ModuleList([nn.LayerNorm(self.hidden, device=dev, dtype=torch.float32) for _ in DEPTHS])\n        self.plain = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)\n        self.centered = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)\n        self.model.requires_grad_(False).eval()\n        for m in (self.norms, self.plain, self.centered):\n            m.requires_grad_(False).eval()\n        self.expected = {}\n        for name, mod in self.loras.items():\n            self.expected[f'encoder.backbone.base_model.model.{name}.lora_A.default.weight'] = mod.a.weight\n            self.expected[f'encoder.backbone.base_model.model.{name}.lora_B.default.weight'] = mod.b.weight\n        for prefix, m in (('norms', self.norms), ('plain', self.plain), ('centered', self.centered)):\n            self.expected.update({f'{prefix}.{n}': p for n, p in m.named_parameters()})\n        assert len(self.expected) == 2 * len(self.loras) + 2 * len(DEPTHS) + 4\n        self.current = None\n\n    def _lora(self, base):\n        torch, nn = self.torch, self.nn\n\n        class TaskLoRA(nn.Module):\n            def __init__(self, layer):\n                super().__init__()\n                self.base_layer = layer\n                dev = layer.weight.device\n                self.a = nn.Linear(layer.in_features, RANK, bias=False, device=dev, dtype=torch.float32)\n                self.b = nn.Linear(RANK, layer.out_features, bias=False, device=dev, dtype=torch.float32)\n\n            def forward(self, x):\n                result = self.base_layer(x)\n                return (result + self.b(self.a(x.float())) * SCALE).to(result.dtype)\n\n        return TaskLoRA(base)\n\n    def load_fold(self, fold):\n        if self.current == fold:\n            return\n        from safetensors.torch import load_file\n        torch = self.torch\n        state = load_file(str(self.assets / f'fold{fold}' / 'trainables.safetensors'))\n        assert set(state) == set(self.expected), (sorted(set(state) - set(self.expected))[:3], sorted(set(self.expected) - set(state))[:3])\n        with torch.inference_mode():\n            for name, param in self.expected.items():\n                assert state[name].shape == param.shape and state[name].dtype == torch.float32, name\n                assert torch.isfinite(state[name]).all(), name\n                param.copy_(state[name].to(param.device))\n        del state\n        gc.collect()\n        self.current = fold\n\n    def _features(self, seqs, idx):\n        torch = self.torch\n        batch = self.tokenizer.pad({'input_ids': [seqs[i] for i in idx]}, padding=True, return_tensors='pt').to('cuda:0')\n        out = self.model(**batch, output_hidden_states=True, use_cache=False, return_dict=True)\n        assert len(out.hidden_states) == self.layers + 1\n        pos = batch['attention_mask'].sum(1) - 1\n        rows = torch.arange(len(idx))\n        parts = []\n        for depth, norm in zip(DEPTHS, self.norms):\n            hs = out.last_hidden_state if depth == self.layers else out.hidden_states[depth]\n            pooled = hs[rows.to(hs.device), pos.to(hs.device)].float()\n            if depth == self.layers:\n                pooled = self.nn.functional.normalize(pooled, dim=-1)\n            parts.append(norm(pooled.to(self.head_device)))\n        return torch.cat(parts, dim=-1)\n\n    def predict(self, texts, folds=range(5), batch_size=4, deadline=None, on_fold=None):\n        torch = self.torch\n        seqs, owners, weights = windows(self.tokenizer, texts)\n        order = sorted(range(len(seqs)), key=lambda i: (len(seqs[i]), seqs[i]))\n        n = len(texts)\n        res = {'plain': [], 'centered': [], 'centered_bias': [], 'folds': []}\n        finite = np.ones(n, dtype=bool)\n        for fold in folds:\n            if deadline is not None and time.time() > deadline:\n                self.log(f'native: deadline reached before fold {fold}')\n                break\n            self.load_fold(fold)\n            t0 = time.time()\n            vals = {'plain': np.zeros((n, 18)), 'centered': np.zeros((n, 18))}\n            with torch.inference_mode():\n                for s in range(0, len(order), batch_size):\n                    if deadline is not None and time.time() >= deadline:\n                        raise TimeoutError('native deadline inside fold; completed-fold checkpoint retained')\n                    idx = order[s:s + batch_size]\n                    try:\n                        feats = self._features(seqs, idx)\n                    except torch.cuda.OutOfMemoryError:\n                        torch.cuda.empty_cache()\n                        feats = torch.cat([self._features(seqs, [i]) for i in idx], 0)\n                    ok = torch.isfinite(feats).all(1).cpu().numpy()\n                    if not ok.all():\n                        for i, good in zip(idx, ok):\n                            if not good:\n                                finite[owners[i]] = False\n                        feats = torch.nan_to_num(feats)\n                    own = owners[idx]\n                    for name, head in (('plain', self.plain), ('centered', self.centered)):\n                        sc = head(feats).float().cpu().numpy().astype(np.float64)\n                        np.add.at(vals[name], own, sc * weights[idx][:, None])\n                    del feats\n            denom = np.bincount(owners, weights=weights, minlength=n)[:, None]\n            for name in ('plain', 'centered'):\n                res[name].append(vals[name] / denom)\n            res['centered_bias'].append(self.centered.bias.detach().cpu().numpy().astype(np.float64).copy())\n            res['folds'].append(int(fold))\n            self.log(f'native: fold {fold} done, {n} rows, {len(seqs)} windows, {time.time() - t0:.0f}s, non-finite rows {int((~finite).sum())}')\n            if on_fold is not None:\n                on_fold({k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite)\n        return {k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite\n\n\ndef save(path, ids, labels, res, finite):\n    tmp = str(path) + '.tmp.npz'\n    np.savez(tmp, ids=np.asarray(ids, dtype=str), labels=np.asarray(labels), finite=finite, **res)\n    os.replace(tmp, path)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument('--base', required=True)\n    p.add_argument('--assets', required=True)\n    p.add_argument('--query', required=True)\n    p.add_argument('--out', required=True)\n    p.add_argument('--work', required=True)\n    p.add_argument('--batch', type=int, default=4)\n    p.add_argument('--deadline', type=float, default=0.0)\n    p.add_argument('--limit', type=int, default=0)\n    a = p.parse_args()\n    T0 = time.time()\n\n    def log(*x):\n        print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)\n\n    frame = pd.read_csv(a.query, dtype={'id': str})\n    assert frame.id.is_unique and len(frame)\n    if a.limit:\n        frame = frame.iloc[:a.limit]\n    texts = frame.text.fillna('').astype(str).tolist()\n    model = NativeQ8(a.base, a.assets, a.work, log=log)\n    log('native: backbone and heads built')\n    folds=[int(x) for x in os.environ.get('NTL_NATIVE_FOLDS','0,1,2').split(',')]\n    res, finite = model.predict(texts, folds=folds, batch_size=a.batch, deadline=a.deadline or None,\n                                on_fold=lambda r, f: save(a.out, frame.id.values, model.labels, r, f))\n    assert len(res['folds']), 'no fold finished'\n    save(a.out, frame.id.values, model.labels, res, finite)\n    log('native: saved', a.out, 'folds', res['folds'].tolist())\n\n\nif __name__ == '__main__':\n    main()\n", 'ntl660_readout.py': 'import json\nimport re\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy.sparse import csr_matrix, diags, hstack\n\nLABELS = sorted([\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n                 \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n                 \'claude-fable-5\', \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\',\n                 \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\'])\nFUNCTION_WORDS = frozenset(\'a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little\'.split())\nVIEWS = [(\'char\', 3, 5, 250000, 1.), (\'word\', 1, 3, 180000, .7),\n         (\'shape\', 2, 6, 60000, .5), (\'function\', 1, 3, 50000, .5)]\n\n\ndef shape_view(text):\n    return re.sub(r\'[^\\W\\d_]+\', lambda m: \'X\' if m[0].isupper() else \'x\', text)\n\n\ndef function_view(text):\n    return \' \'.join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else \'CONTENT\'\n                    for t in re.findall(r\'\\w+|[^\\w\\s]\', text))\n\n\ndef prepare(texts, view, keep=()):\n    if view == \'shape\':\n        return [shape_view(t) for t in texts]\n    if view == \'function\':\n        return [function_view(t) for t in texts]\n    if view == \'distort\':\n        keep = set(keep)\n        return [re.sub(r"[A-Za-z’\']+", lambda m: m[0] if m[0].lower() in keep else \'*\', t) for t in texts]\n    return texts\n\n\ndef log_softmax(z):\n    z = z - z.max(1, keepdims=True)\n    return z - np.log(np.exp(z).sum(1, keepdims=True))\n\n\ndef softmax(z):\n    z = z - z.max(1, keepdims=True)\n    z = np.exp(z)\n    return z / z.sum(1, keepdims=True)\n\n\ndef sparse_group_centre(X, groups, lo=2, hi=18):\n    X = csr_matrix(X)\n    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)\n    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))\n    if not len(keep):\n        return X.copy()\n    col = np.full(len(cnt), -1)\n    col[keep] = np.arange(len(keep))\n    rows = np.flatnonzero(col[inv] >= 0)\n    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))\n    means = diags(1.0 / cnt[keep]) @ (M.T @ X)\n    return (X - M @ means).tocsr()\n\n\ndef style_groups(texts):\n    from scipy.cluster.hierarchy import fcluster, linkage\n    from scipy.spatial.distance import squareform\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    if len(texts) < 2:\n        return np.ones(len(texts), dtype=int)\n    try:\n        X = TfidfVectorizer(stop_words=\'english\', sublinear_tf=True, max_features=100000,\n                            dtype=np.float32).fit_transform(texts)\n        D = np.clip(1.0 - (X @ X.T).toarray(), 0., 1.)\n    except ValueError as exc:\n        if \'empty vocabulary\' not in str(exc):\n            raise\n        D = np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)\n    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)\n    D = np.asarray(D / med, dtype=np.float64).copy()\n    np.fill_diagonal(D, 0.)\n    return fcluster(linkage(squareform(D, checks=False), method=\'average\'), .90, criterion=\'distance\')\n\n\nclass Readout:\n    def __init__(self, folder):\n        self.folder = Path(folder)\n        self.meta = json.loads((self.folder / \'metadata.json\').read_text())\n        if self.meta[\'labels\'] != LABELS:\n            raise ValueError(\'Unexpected canonical class order\')\n        self.vectorizers = [joblib.load(self.folder / e[\'file\']) for e in self.meta[\'views\']]\n        from sklearn.feature_extraction.text import TfidfVectorizer\n        assert all(type(v) is TfidfVectorizer and not callable(v.analyzer) and\n                   v.preprocessor is None and v.tokenizer is None for v in self.vectorizers)\n\n    def matrix(self, texts):\n        return hstack([v.transform(prepare(texts, e[\'view\'], self.meta.get(\'keep\', []))) * e[\'weight\']\n                       for v, e in zip(self.vectorizers, self.meta[\'views\'])], format=\'csr\')\n\n    def raw(self, X, state=\'model\'):\n        with np.load(self.folder / (state + \'.npz\'), allow_pickle=False) as z:\n            scores = np.asarray(X @ z[\'coef\'].T) + z[\'intercept\']\n            classes = z[\'classes\'].copy()\n        if len(classes) == 2 and scores.shape[1] == 1:\n            scores = np.column_stack([-scores[:, 0], scores[:, 0]])\n        return scores, classes\n\n    def scores(self, texts, state=\'model\'):\n        return self.raw(self.matrix(texts), state)\n\n\ndef core_votes(root, texts):\n    votes = {}\n    for key, weight in [(\'svc4\', 12.0), (\'tfidf\', .25), (\'distort\', .10)]:\n        r = Readout(Path(root) / \'core\' / key)\n        scores, classes = r.scores(texts)\n        assert classes.tolist() == list(range(len(LABELS)))\n        votes[key] = log_softmax(scores) * weight\n    return votes\n\n\ndef complementary_style(root, texts):\n    root = Path(root)\n    r = Readout(root / \'comp\' / \'stylegram\')\n    X = r.matrix(texts)\n    d, classes = r.raw(X, \'plain\')\n    assert classes.tolist() == list(range(len(LABELS)))\n    P = softmax(d)\n    groups = style_groups(texts)\n    sizes = np.bincount(groups)[groups]\n    rows = (sizes >= 2) & (sizes <= 18)\n    if rows.any():\n        d, _ = r.raw(sparse_group_centre(X[rows], groups[rows]), \'twin\')\n        P[rows] = softmax(d)\n    gates = {}\n    for family in [\'claude\', \'gpt\', \'gemini\', \'glm\']:\n        g = Readout(root / \'comp\' / \'gates\' / family)\n        d, classes = g.scores(texts)\n        expected = [i for i, label in enumerate(LABELS) if label.startswith(family)]\n        assert classes.tolist() == expected\n        gates[family] = d\n    return P, gates, groups\n', 'pipe.py': 'import gc\nimport os\nimport threading\nimport time\nfrom collections import deque\n\nimport numpy as np\n\n\ndef _flat(x):\n    if isinstance(x, dict):\n        parts = [x[k] for k in sorted(x)]\n    else:\n        parts = x if isinstance(x, (tuple, list)) else [x]\n    return np.concatenate([np.ravel(np.asarray(p, dtype=np.float64)) for p in parts])\n\n\ndef _agree(batches, work, workers, tag, log):\n    probe = [batches[len(batches) // 2 + i] for i in range(workers)]\n    try:\n        ref = [work(b) for b in probe]\n        if any(r is None for r in ref):\n            return False\n        got = [None] * workers\n        err = []\n\n        def go(i):\n            try:\n                got[i] = work(probe[i])\n            except Exception as e:\n                err.append(e)\n\n        threads = [threading.Thread(target=go, args=(i,)) for i in range(workers)]\n        [t.start() for t in threads]\n        [t.join() for t in threads]\n        ok = not err and all(g is not None and np.abs(_flat(r) - _flat(g)).max() <= 2e-2 * (np.abs(_flat(r)).mean() + 1e-6) for r, g in zip(ref, got))\n    except Exception as e:\n        log(f\'{tag}: concurrent check raised {e!r:.100}\')\n        ok = False\n    log(f\'{tag}: concurrent check {"passed" if ok else "failed, single worker"}\')\n    return ok\n\n\ndef pipelined(batches, work, store, fallback, workers=2, deadline=None, tag=\'pipe\', log=print):\n    batches = list(batches)\n    t0 = time.time()\n    workers = min(workers, int(os.environ.get(\'NTL_PIPE_WORKERS\', workers)))\n    if workers < 2 or len(batches) < 6 or not _agree(batches, work, workers, tag, log):\n        workers = 1\n    queue = deque(batches)\n    redo = []\n    lock = threading.Lock()\n    stop = threading.Event()\n\n    def loop():\n        while not stop.is_set():\n            if deadline is not None and time.time() >= deadline:\n                return\n            with lock:\n                if not queue:\n                    return\n                b = queue.popleft()\n            try:\n                r = work(b)\n            except Exception:\n                with lock:\n                    redo.append(b)\n                stop.set()\n                return\n            with lock:\n                if r is None:\n                    redo.append(b)\n                else:\n                    store(b, r)\n\n    if workers == 1:\n        while queue and not (deadline is not None and time.time() >= deadline):\n            b = queue.popleft()\n            r = work(b)\n            if r is None:\n                redo.append(b)\n            else:\n                store(b, r)\n    else:\n        threads = [threading.Thread(target=loop) for _ in range(workers)]\n        [t.start() for t in threads]\n        [t.join() for t in threads]\n    rest = redo + list(queue)\n    if rest:\n        gc.collect()\n        try:\n            import torch\n            torch.cuda.empty_cache()\n        except Exception:\n            pass\n    for b in rest:\n        if deadline is not None and time.time() >= deadline:\n            break\n        fallback(b)\n    log(f\'{tag}: {len(batches)} batches, {workers} worker(s), {len(redo)} redone sequentially, {time.time() - t0:.0f}s\')\n\n\ndef spans_devices(model):\n    return len({p.device for p in model.parameters()}) > 1\n', 'qwen_groups.py': "import argparse\nimport json\nimport os\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom transformers import AutoModel, AutoTokenizer\n\n\ndef stage(base, assets, work):\n    base, assets, staged = Path(base), Path(assets), Path(work) / 'qgroup-base'\n    staged.mkdir(parents=True, exist_ok=True)\n    index = json.loads((assets / 'base' / 'model.safetensors.index.json').read_text())\n    for name in ('config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json'):\n        target = staged / name\n        if not target.exists():\n            shutil.copyfile(assets / 'base' / name, target)\n    for name in sorted(set(index['weight_map'].values())):\n        source, target = base / name, staged / name\n        assert source.is_file(), str(source)\n        if not target.exists():\n            try:\n                os.symlink(source, target)\n            except OSError:\n                shutil.copyfile(source, target)\n    return staged\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    for key in ('base', 'assets', 'query', 'out', 'work'):\n        p.add_argument('--' + key, required=True)\n    p.add_argument('--max_len', type=int, default=512)\n    p.add_argument('--tok_budget', type=int, default=8192)\n    p.add_argument('--mem_frac', type=float, default=0.88)\n    a = p.parse_args()\n    started = time.time()\n    frame = pd.read_csv(a.query, dtype={'id': str})\n    texts = frame.text.fillna('').astype(str).tolist()\n    staged = stage(a.base, a.assets, a.work)\n    tok = AutoTokenizer.from_pretrained(staged, local_files_only=True)\n    auto, plain = tok('hello')['input_ids'], tok('hello', add_special_tokens=False)['input_ids']\n    eos = auto[-1] if (len(auto) > len(plain) and auto[-1] != plain[-1]) else (tok.eos_token_id if tok.eos_token_id is not None else 151643)\n    pad = tok.pad_token_id if tok.pad_token_id is not None else eos\n    gpus = torch.cuda.device_count()\n    assert gpus >= 1\n    max_memory = {i: f'{int(torch.cuda.get_device_properties(i).total_memory * a.mem_frac / 2 ** 30)}GiB' for i in range(gpus)}\n    model = AutoModel.from_pretrained(staged, local_files_only=True, dtype=torch.float16, device_map='auto', max_memory=max_memory).eval()\n    first = next(model.parameters()).device\n    ids = []\n    for t in texts:\n        x = tok(t, truncation=True, max_length=a.max_len - 1)['input_ids']\n        ids.append(x if x[-1] == eos else x + [eos])\n    order = np.argsort([len(x) for x in ids], kind='stable')\n    emb = np.zeros((len(texts), model.config.get_text_config().hidden_size), np.float16)\n    budget, done, ooms = a.tok_budget, 0, 0\n    with torch.inference_mode():\n        while done < len(order):\n            end = done + 1\n            while end < len(order) and (end - done + 1) * len(ids[order[end]]) <= budget:\n                end += 1\n            idx = order[done:end]\n            try:\n                width = max(len(ids[i]) for i in idx)\n                inp = torch.full((len(idx), width), pad)\n                att = torch.zeros((len(idx), width), dtype=torch.long)\n                for r, i in enumerate(idx):\n                    inp[r, :len(ids[i])] = torch.tensor(ids[i])\n                    att[r, :len(ids[i])] = 1\n                hidden = model(input_ids=inp.to(first), attention_mask=att.to(first), use_cache=False).last_hidden_state\n                vec = hidden[torch.arange(len(idx), device=hidden.device), (att.sum(1) - 1).to(hidden.device)].float()\n                emb[idx] = torch.nn.functional.normalize(vec, dim=-1).cpu().numpy().astype(np.float16)\n                done = end\n            except torch.cuda.OutOfMemoryError:\n                torch.cuda.empty_cache()\n                ooms += 1\n                budget = max(len(ids[order[done]]), budget // 2)\n                print(f'qwen_groups: OOM, token budget -> {budget}', flush=True)\n    assert np.isfinite(emb).all()\n    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)\n    np.save(a.out, emb)\n    print(f'qwen_groups: {len(texts)} rows, dim {emb.shape[1]}, {ooms} OOM retries, {time.time() - started:.0f} s', flush=True)\n\n\nif __name__ == '__main__':\n    main()\n", 'run_optimized.py': '"""Bounded inference reproduction of ozertuu v37; no new training claim."""\nimport glob\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport sys\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom budget import atomic_json, run_stage\nfrom ensemble import LABELS, checkpoint, combine, decide_scores, load_member, softmax\n\nT0 = time.monotonic()\nWORK=Path(\'/kaggle/working\')\nTEMP=Path(\'/kaggle/temp/ntl-optimized\')\nTEMP.mkdir(parents=True,exist_ok=True)\nOUTPUT=WORK/\'submission.parquet\'\nREPORT=WORK/\'runtime-report.json\'\nPROGRESS=WORK/\'progress\'\nPROGRESS.mkdir(exist_ok=True)\nDATA=next(Path(p) for p in [\'/kaggle/input/competitions/name-that-llm\',\'/kaggle/input/name-that-llm\'] if (Path(p)/\'train.csv\').exists())\ntrain=pd.read_csv(DATA/\'train.csv\',dtype={\'id\':str})\nactual=pd.read_csv(DATA/\'test.csv\',dtype={\'id\':str})\nassert actual.id.is_unique and len(actual)\nis_benchmark=len(actual)<50\n# Public commit runs real inference on 72 length-stratified texts plus the actual sample.\n# No accuracy is reported: the published heads may have seen official training rows.\nif is_benchmark:\n    ordered=train.assign(_length=train.text.fillna(\'\').str.len()).sort_values(\'_length\',kind=\'stable\')\n    pos=np.linspace(0,len(ordered)-1,72,dtype=int)\n    bench=ordered.iloc[pos][[\'text\']].copy().reset_index(drop=True)\n    bench[\'id\']=[f\'timing_{i:04d}\' for i in range(len(bench))]\n    test=pd.concat([actual[[\'id\',\'text\']],bench[[\'id\',\'text\']]],ignore_index=True)\nelse:\n    test=actual[[\'id\',\'text\']].copy()\ntest[\'text\']=test.text.fillna(\'\').astype(str)\ndata=TEMP/\'data\';data.mkdir(exist_ok=True)\ntrain.to_csv(data/\'train.csv\',index=False)\ntest.to_csv(data/\'test.csv\',index=False)\nids=test.id.tolist()\nprior=np.bincount(train.model.map({l:i for i,l in enumerate(LABELS)}),minlength=18)+1.\nreport=dict(account=\'liuqingmu096\',source=\'ozertuu Version 37\',is_timing_benchmark=is_benchmark,\n    official_test_rows=len(actual),inference_rows=len(test),benchmark_accuracy_not_validated=True,\n    removed=[\'complementary stack\',\'Gaussian extra train inference\',\'GSQ 27B\',\'GLM likelihood reranker\'],\n    main_adapters_per_family=3,native_folds_requested=3,stages=[],completed_members=[],native_folds=[],\n    elapsed_seconds=0.,mode=\'starting\')\nTOTAL=6600 if is_benchmark else 7.5*3600\n# Deadline includes setup, CPU baseline, model loads, and every child process.\ndeadline=T0+TOTAL\nenv=dict(os.environ,HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',TOKENIZERS_PARALLELISM=\'false\',\n         PYTHONUNBUFFERED=\'1\',NTL_PIPE_WORKERS=\'1\',OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\')\n\n\ndef save_report():\n    report[\'elapsed_seconds\']=time.monotonic()-T0\n    atomic_json(REPORT,report)\n\n\ndef first(pattern):\n    hits=sorted(glob.glob(\'/kaggle/input/**/\'+pattern,recursive=True))\n    assert hits,pattern\n    return hits[0]\n\n\ndef stage(name,cmd,maximum,minimum=10):\n    left=deadline-time.monotonic()-120\n    result=run_stage(name,cmd,min(maximum,max(0,left)),PROGRESS,env) if left>=minimum else dict(name=name,status=\'skipped_budget\',seconds=0.)\n    report[\'stages\'].append(result);save_report()\n    return result\n\n\n# Resolve and extract only the original prefit archive; keep original path guards.\nroot=Path(first(\'ntl666-cache-parts.json\')).parent\nparts=json.loads((root/\'ntl666-cache-parts.json\').read_text())\nassert parts[\'format\']==\'ntl666-cache-parts-v1\'\nassert [p[\'name\'] for p in parts[\'parts\']]==[f\'ntl666-prefit.part{i:03d}.bin\' for i in range(len(parts[\'parts\']))]\narchive=TEMP/\'prefit.zip\'\nwith archive.open(\'wb\') as target:\n    for part in parts[\'parts\']:\n        source=root/part[\'name\'];assert source.stat().st_size==part[\'bytes\']\n        with source.open(\'rb\') as stream:shutil.copyfileobj(stream,target,16*1024*1024)\nassert archive.stat().st_size==parts[\'archive_bytes\']\nwith zipfile.ZipFile(archive) as z:\n    entries=z.infolist()\n    assert len(entries)==parts[\'files\'] and len({e.filename for e in entries})==len(entries)\n    for e in entries:\n        p=PurePosixPath(e.filename)\n        assert not p.is_absolute() and \'..\' not in p.parts and chr(92) not in e.filename and p.parts[0]==\'ntl666-prefit\' and not e.is_dir()\n    assert z.read(\'ntl666-prefit/COMPLETE\')==b\'ntl666-v14-data-numeric-v1\\n\'\n    provenance=json.loads(z.read(\'ntl666-prefit/provenance.json\'))\n    assert provenance[\'status\']==\'COMPLETE\' and provenance[\'pool_rows\']==190588 and provenance[\'dataset_version\']==13\n    z.extractall(TEMP)\narchive.unlink()\nprefit=TEMP/\'ntl666-prefit\'\nfrom ntl660_readout import core_votes\nterms=core_votes(prefit,test.text.tolist())\nassert all(x.shape==(len(test),18) and np.isfinite(x).all() for x in terms.values())\nscores=combine(terms,prior)\ncheckpoint(OUTPUT,ids,scores)\nreport[\'mode\']=\'prefit_baseline\';report[\'setup_seconds\']=time.monotonic()-T0\nsave_report()\nprint(\'BASELINE_CHECKPOINT_READY\',len(ids),\'rows\',flush=True)\n\n# These workers use full-precision published backbones, not GSQ/BNB quantization.\n# Do not reinstall duplicate offline wheels from multiple mounted datasets.\nconfigs=sorted(glob.glob(\'/kaggle/input/**/config.json\',recursive=True))\ndef model(*parts):\n    hits=[str(Path(c).parent) for c in configs if all(p in c.lower() for p in parts)]\n    assert hits,parts\n    return hits[0]\ndef adapter(name):return str(Path(first(name+\'/heads.pt\')).parent)\ndef meta(name):\n    folder=Path(first(name+\'/meta.json\'));return str(folder.parent),json.loads(folder.read_text())[\'cfg\']\nB8=model(\'/qwen-3/\',\'/8b-base/\')\nB14=model(\'/qwen-3/\',\'/14b-base/\')\nBL=model(\'/llama-3-1-8b-base/\')\nB25=model(\'/qwen2.5/\',\'/14b/\')\nNB=model(\'/qwen-3-embedding/\',\'/8b/\')\nNATIVE=str(Path(first(\'ntl-native-q8-multidepth-fivefold-0679/manifest.json\')).parent)\nN14,N14C=meta(\'lora_qwen3_14b\')\nQ25,Q25C=meta(\'lora_qwen25_14b\')\nOQ4=str(Path(first(\'ntl-q14-lora-v4/heads.safetensors\')).parent)\nDEB=str(Path(first(\'deberta_v3_large/meta.json\')).parent)\nL=[2,4,6,8,12,16,20,24]\nspecs=[\n dict(name=\'deb\',type=\'enc\',ckpt=DEB,w=1.),\n dict(name=\'q8v5\',type=\'l24\',path=B8,adapters=[adapter(\'q8v5\'),adapter(\'q8v5_f0\'),adapter(\'q8v5_f1\')],keep=24,layers=L,w=2.,bs=16,max_len=384,twin=.35,dp=True),\n dict(name=\'q14v5\',type=\'l24\',path=B14,adapters=[adapter(\'q14v5\'),adapter(\'q14v5_f0\'),adapter(\'q14v5_f1\')],keep=27,layers=L,w=3.,bs=8,max_len=384,twin=.35),\n dict(name=\'n14b_clean\',type=\'l24\',path=B14,adapters=[OQ4,N14],adapter_weights=[.72,.28],keep=N14C[\'keep\'],layers=N14C[\'layers\'],w=2.5,bs=8,max_len=N14C[\'max_len\'],twin=.35,no_final_norm=True,eos_id=N14C[\'eos_id\']),\n dict(name=\'ll8v5\',type=\'l24\',path=BL,adapters=[adapter(\'ll8v5\'),adapter(\'ll8v5_f0\'),adapter(\'ll8v5_f1\')],keep=24,layers=L,w=1.,bs=16,max_len=384,twin=.35,dp=True),\n dict(name=\'q25_14\',type=\'l24\',path=B25,adapters=[Q25],keep=Q25C[\'keep\'],layers=Q25C[\'layers\'],w=1.,bs=8,max_len=Q25C[\'max_len\'],twin=.35,no_final_norm=True,eos_id=Q25C[\'eos_id\'])]\nreport[\'specs\']=specs;save_report()\n\n# Cache semantic groups once; all members and native calibration use this same grouping.\ngroup_file=TEMP/\'groups.npy\'\nembed=TEMP/\'group_emb.npy\'\ng=stage(\'semantic-groups\',[sys.executable,\'qwen_groups.py\',\'--base\',NB,\'--assets\',NATIVE,\'--query\',str(data/\'test.csv\'),\'--out\',str(embed),\'--work\',str(TEMP/\'groups\')],420 if is_benchmark else 1200)\nif g[\'status\']==\'complete\' and embed.exists():\n    env[\'NTL_GROUP_EMBED\']=str(embed)\nfor spec in specs:\n    name=spec[\'name\'];member=TEMP/(name+\'.npz\')\n    command=[sys.executable,\'infer_worker.py\',\'--data\',str(data),\'--models\',json.dumps([spec]),\n        \'--tfidf_w\',\'0\',\'--distort_w\',\'0\',\'--svc_w\',\'0\',\'--adapt_w\',\'1\',\'--adapt_thr\',\'.15\',\n        \'--sinkhorn\',\'.30\',\'--sinkhorn_target_mix\',\'.35\',\'--out\',str(TEMP/(name+\'.parquet\'))]\n    env[\'NTL_MEMBER_OUTPUT\']=str(member);env[\'NTL_GROUP_OUTPUT\']=str(group_file)\n    limit=(600 if name not in (\'q14v5\',\'n14b_clean\') else 780) if is_benchmark else (900 if name==\'deb\' else 3900)\n    s=stage(name,command,limit)\n    if s[\'status\']==\'complete\' and member.exists():\n        try:\n            values=load_member(member,ids)\n            assert name in values\n            terms.update(values)\n            scores=combine(terms,prior)\n            checkpoint(OUTPUT,ids,scores)\n            report[\'completed_members\'].append(name);report[\'mode\']=\'main_ensemble\';save_report()\n            print(\'ENSEMBLE_CHECKPOINT\',name,len(ids),\'rows\',flush=True)\n        except Exception as e:\n            s[\'validation_error\']=repr(e);save_report()\n\n# Native saves every complete fold atomically. A killed later fold cannot erase prior folds.\nnative=TEMP/\'native.npz\'\nenv[\'NTL_NATIVE_FOLDS\']=\'0,1,2\'\nstage(\'native\',[sys.executable,\'native_q8.py\',\'--base\',NB,\'--assets\',NATIVE,\'--query\',str(data/\'test.csv\'),\n    \'--out\',str(native),\'--work\',str(TEMP/\'native\'),\'--deadline\',str(time.time()+max(0,deadline-time.monotonic()-180))],900 if is_benchmark else 6000)\nif native.exists() and report[\'completed_members\'] and group_file.exists():\n    try:\n        import native_q8\n        with np.load(native,allow_pickle=False) as z:\n            assert z[\'ids\'].astype(str).tolist()==ids\n            labels=z[\'labels\'].astype(str).tolist();order=[labels.index(l) for l in LABELS]\n            folds=z[\'folds\'].tolist();assert folds and len(folds)<=3\n            states=[json.loads((Path(NATIVE)/\'calibration\'/f\'fold{k}.json\').read_text())[\'models\'][\'qwen8\'] for k in folds]\n            groups=np.load(group_file,allow_pickle=False);assert groups.shape==(len(ids),)\n            lp=native_q8.calibrate(z[\'plain\'][:,:,order],z[\'centered\'][:,:,order],z[\'centered_bias\'][:,order],groups,states,True)\n            agrees=[float((x.argmax(1)==scores.argmax(1)).mean()) for x in lp]\n            assert min(agrees)>=.35,(\'native agreement guard\',agrees)\n            share=(.40*z[\'finite\'])[:,None]\n            active=sum(float(abs(v).max())>0 for v in terms.values())\n            base=softmax(sum(terms.values())/max(1,active))\n            mixed=sum(softmax(decide_scores(softmax((1-share)*np.log(np.maximum(base,1e-15))+share*x),prior)) for x in lp)/len(lp)\n            scores=np.log(np.maximum(mixed,1e-15))\n            checkpoint(OUTPUT,ids,scores)\n            report[\'native_folds\']=folds;report[\'native_agreement\']=agrees;report[\'mode\']=\'main_plus_native\'\n    except Exception as e:report[\'native_validation_error\']=repr(e)\n\nif is_benchmark:\n    full=pd.read_parquet(OUTPUT)\n    full.to_parquet(WORK/\'timing-predictions.parquet\',index=False)\n    checkpoint(OUTPUT,actual.id.tolist(),scores[:len(actual)])\nfinal=pd.read_parquet(OUTPUT)\nassert final.columns.tolist()==[\'id\',\'model\'] and final.id.tolist()==actual.id.tolist()\nassert final.id.is_unique and final.model.isin(LABELS).all()\nreport[\'final_rows\']=len(final);report[\'final_valid\']=True\nreport[\'estimated_4000_inference_seconds\']={s[\'name\']:s[\'seconds\']*4000/len(test) for s in report[\'stages\'] if s[\'status\']==\'complete\' and s[\'name\'] not in (\'offline-wheel\',)}\nsave_report()\nprint(\'FINAL_VALIDATED\',json.dumps(report),flush=True)\n', 'self_check.py': '"""Exercise the production POSIX process-tree watchdog before loading any GPU model."""\nimport os\nfrom pathlib import Path\nimport sys\nimport signal\nimport subprocess\nimport tempfile\nimport time\nimport numpy as np\nfrom budget import run_stage\nfrom ensemble import checkpoint\n\nassert os.name==\'posix\',\'Production watchdog requires Kaggle Linux process groups\'\nwith tempfile.TemporaryDirectory() as tmp:\n    root=Path(tmp)\n    marker=root/\'grandchild-survived\'\n    child="import time;from pathlib import Path;time.sleep(1);Path("+repr(str(marker))+").write_text(\'survived\')"\n    parent="import subprocess,sys,time;subprocess.Popen([sys.executable,\'-c\',"+repr(child)+"],start_new_session=True);print(\'READY\',flush=True);time.sleep(5)"\n    # Red: the previous process-group-only kill leaves an independently grouped\n    # nested stage alive. Its short-lived fixture writes the marker and exits.\n    old=subprocess.Popen([sys.executable,\'-c\',parent],stdout=subprocess.PIPE,text=True,start_new_session=True)\n    assert old.stdout.readline().strip()==\'READY\'\n    os.killpg(old.pid,signal.SIGKILL);old.wait(timeout=5)\n    time.sleep(1.2)\n    assert marker.exists(),\'red fixture did not reproduce nested-session escape\'\n    marker.unlink()\n    print(\'RED_CONFIRMED_NESTED_SESSION_ESCAPED_GROUP_ONLY_KILL\',flush=True)\n    pred=root/\'submission.parquet\'\n    checkpoint(pred,[\'a\',\'b\'],np.arange(36,dtype=float).reshape(2,18))\n    original=pred.read_bytes()\n    result=run_stage(\'tree\',[sys.executable,\'-c\',parent],.25,root)\n    assert result[\'status\']==\'timeout\' and result[\'seconds\']<2\n    time.sleep(1.1)\n    assert not marker.exists(),\'grandchild escaped timeout\'\n    assert pred.read_bytes()==original,\'completed prediction lost\'\nprint(\'POSIX_PROCESS_TREE_TIMEOUT_AND_CHECKPOINT_PASSED\',flush=True)\n'}
for name,content in FILES.items():
    pathlib.Path(name).write_text(content,encoding='utf-8')
sys.path.insert(0,os.getcwd())
from budget import run_stage, atomic_json
import pandas as pd
import torch
gpu=dict(torch_version=torch.__version__,cuda_version=torch.version.cuda,available=torch.cuda.is_available(),device_count=torch.cuda.device_count())
assert gpu['cuda_version'] and gpu['available'] and gpu['device_count']==2, gpu
gpu['devices']=[torch.cuda.get_device_name(i) for i in range(2)]
for i in range(2):
    with torch.cuda.device(i):
        x=torch.ones((64,64),device=f'cuda:{i}')
        assert float((x@x)[0,0].item())==64.
        del x
atomic_json('/kaggle/working/gpu-validation.json',dict(ok=True,**gpu))
print('PYTORCH_GPU_VALIDATED',json.dumps(gpu),flush=True)
selfcheck=run_stage('self-check',[sys.executable,'self_check.py'],30,'/kaggle/working')
assert selfcheck['status']=='complete',selfcheck
data=next(pathlib.Path(p) for p in ['/kaggle/input/competitions/name-that-llm','/kaggle/input/name-that-llm'] if (pathlib.Path(p)/'test.csv').exists())
actual=pd.read_csv(data/'test.csv',dtype={'id':str})
benchmark=len(actual)<50
result=run_stage('pipeline',[sys.executable,'-u','run_optimized.py'],6900 if benchmark else 7.8*3600,'/kaggle/working',dict(os.environ,PYTHONUNBUFFERED='1'))
out=pathlib.Path('/kaggle/working/submission.parquet')
assert out.exists(),result
frame=pd.read_parquet(out)
assert frame.id.is_unique and frame.model.notna().all()
assert set(actual.id)<=set(frame.id)
frame=frame.set_index('id').loc[actual.id].reset_index()
assert frame.columns.tolist()==['id','model'] and frame.id.tolist()==actual.id.tolist()
temp=out.with_name('submission.final.tmp.parquet')
frame.to_parquet(temp,index=False)
os.replace(temp,out)
atomic_json('/kaggle/working/outer-validation.json',dict(valid=True,rows=len(frame),pipeline=result,public_timing_benchmark=benchmark))
print('OUTER_VALIDATED',len(frame),'rows',result,flush=True)
